In [ ]:
from pyscf import gto, scf
from qarp.operators import JordanWigner
from qarp.operators.integrals import restricted_integrals_to_fermion_operator
from qarp.operators.ucc import ucc_singles_and_doubles
from qarp.operators.pyscf import active_space_from_mf

mol = gto.M(atom="H 0 0 0; Li 0 0 1.59", basis="sto3g", symmetry=True, verbose=3)

mol.build()
mf = scf.RHF(mol)
mf.kernel()

integrals, onv = active_space_from_mf(mf, 2, 3)
fermion_operator = restricted_integrals_to_fermion_operator(*integrals)
qop = JordanWigner().encode_operator(fermion_operator)
fucc = ucc_singles_and_doubles(onv, generalised=False, spin_conserving=False)[0]
qucc = JordanWigner().encode_operator(fucc)
print(onv)

In [ ]:
from qarp.algorithms import AdaptVQE
from qarp.algorithms import StateVector
from qarp.blocks import ComputationalBasisStateBlock

ref = ComputationalBasisStateBlock(onv)
ref.build()

adapt = AdaptVQE(
    reference_block=ref,
    system_hamiltonian=qop,
    excitation_pool=qucc,
    primitive=StateVector(),
    gradient=True,
    gradient_thresh=1e-9,
    convergence_thresh=1e-10,
    exc_per_iter=1
)
adapt.build()
s0_en, _ = adapt.run()

In [ ]:
from qarp.algorithms import AdaptVQD

adapt = AdaptVQE(
    reference_block=ref,
    system_hamiltonian=qop,
    excitation_pool=qucc,
    primitive=StateVector(),
    gradient=False
)
adapt.build()
s0_en, _ = adapt.run()

In [ ]:
ground_state = adapt.get_final_state_block()
adaptvqd = AdaptVQD(
        reference_block=ref, 
        hamiltonian=qop, 
        excitation_pool=qucc, 
        orthogonal_states=[adapt.get_final_state_block()], 
        betas=[5], 
        exc_per_iter=2,
        gradient=False)
adaptvqd.gradient_thresh = 1e-4
adaptvqd.build()
s1_en, _ = adaptvqd.run()